In [ ]:
import cv2
import easyocr
import re
from typing import List, Dict

In [ ]:
# Initialiser le lecteur OCR (une seule fois)
reader = easyocr.Reader(['en'], gpu=False)

In [ ]:
def extract_text_from_image(image_path: str) -> List:
    """
    Extrait tout le texte d'une image F1
    """
    image = cv2.imread(image_path)
    results = reader.readtext(image)
    return results


[ WARN:0@268.762] global loadsave.cpp:278 findDecoder imread_('../data/image.png'): can't open/read file: check file path/integrity


In [ ]:
def parse_driver_data(ocr_results: List) -> List[Dict]:
    """
    Parse les résultats OCR pour extraire position, pilote, intervalle
    """
    drivers = []
    current_driver = {}
    
    for (bbox, text, prob) in ocr_results:
        text = text.strip()
        
        # Position (1, 2, 3, ...)
        if text.isdigit() and len(text) <= 2:
            if current_driver:
                drivers.append(current_driver)
            current_driver = {'position': int(text)}
        
        # Code pilote (VER, HAM, LEC, ...)
        elif re.match(r'^[A-Z]{3}$', text):
            current_driver['driver'] = text
        
        # Intervalle (+2.530 H, Interval M, ...)
        elif '+' in text or 'Interval' in text:
            current_driver['interval'] = text
    
    # Ajouter le dernier pilote
    if current_driver:
        drivers.append(current_driver)
    
    return drivers


In [ ]:
def get_f1_data(image_path: str) -> List[Dict]:
    """
    Fonction principale : donne le chemin de l'image, récupère les données
    """
    ocr_results = extract_text_from_image(image_path)
    drivers_data = parse_driver_data(ocr_results)
    return drivers_data

FileNotFoundError: ../data/image.png does not exist

In [ ]:
def save_data_to_dict(drivers_data: List[Dict]) -> Dict:
    """
    Transforme les données en dictionnaire structuré
    """
    return {
        "total_drivers": len(drivers_data),
        "drivers": drivers_data,
        "leader": drivers_data[0] if drivers_data else None
    }


In [ ]:
# Affichage avec Matplotlib
results[0].show()  # Affiche l'image avec les boîtes


